In [5]:
import os
import requests
import pandas as pd
from datetime import date

# === CONFIG ===
target_date = date.today().isoformat()
folder = "data/lineups"
os.makedirs(folder, exist_ok=True)
filename = f"lineups_statsapi_{target_date}.csv"
output_path = os.path.join(folder, filename)

# === GET GAME IDS FOR THE DAY ===
schedule_url = f"https://statsapi.mlb.com/api/v1/schedule?sportId=1&date={target_date}"
schedule_resp = requests.get(schedule_url)
schedule = schedule_resp.json()

game_pks = []
for d in schedule.get("dates", []):
    for game in d.get("games", []):
        game_pks.append(game["gamePk"])

# === PARSE LINEUPS FOR EACH GAME ===
all_rows = []

for game_pk in game_pks:
    game_url = f"https://statsapi.mlb.com/api/v1.1/game/{game_pk}/feed/live"
    game_resp = requests.get(game_url)
    if game_resp.status_code != 200:
        continue
    game_data = game_resp.json()

    boxscore = game_data.get("liveData", {}).get("boxscore", {})
    players = game_data.get("liveData", {}).get("players", {})
    game_info = game_data.get("gameData", {})

    away_team = game_info.get("teams", {}).get("away", {}).get("name")
    home_team = game_info.get("teams", {}).get("home", {}).get("name")

    try:
        away_starter_id = boxscore["teams"]["away"]["info"][0]["fieldList"][0]["person"]["id"]
        home_starter_id = boxscore["teams"]["home"]["info"][0]["fieldList"][0]["person"]["id"]
    except:
        away_starter_id = home_starter_id = None

    away_pitcher = players.get(f"ID{away_starter_id}", {}).get("person", {}).get("fullName") if away_starter_id else None
    home_pitcher = players.get(f"ID{home_starter_id}", {}).get("person", {}).get("fullName") if home_starter_id else None

    for team_side in ["away", "home"]:
        team_name = away_team if team_side == "away" else home_team
        team_batters = boxscore["teams"][team_side].get("battingOrder", [])
        for i, player_id in enumerate(team_batters):
                player_key = f"ID{player_id}"
                player = players.get(player_key, {})

                full_name = player.get("fullName") or player.get("person", {}).get("fullName")
                pos = player.get("position", {}).get("abbreviation")

                all_rows.append({
                "Date": target_date,
                "Team": team_name,
                "Side": team_side,
                "Opponent": home_team if team_side == "away" else away_team,
                "Pitcher": away_pitcher if team_side == "away" else home_pitcher,
                "Batting_Order": i + 1,
                "Player": full_name,
                "Position": pos
            })


# === SAVE TO CSV ===
df = pd.DataFrame(all_rows)
df.to_csv(output_path, index=False)
print(f"✅ Lineups saved to {output_path}")
print(df.head())


✅ Lineups saved to data/lineups/lineups_statsapi_2025-05-25.csv
         Date                 Team  Side        Opponent Pitcher  \
0  2025-05-25  Cleveland Guardians  away  Detroit Tigers    None   
1  2025-05-25  Cleveland Guardians  away  Detroit Tigers    None   
2  2025-05-25  Cleveland Guardians  away  Detroit Tigers    None   
3  2025-05-25  Cleveland Guardians  away  Detroit Tigers    None   
4  2025-05-25  Cleveland Guardians  away  Detroit Tigers    None   

   Batting_Order Player Position  
0              1   None     None  
1              2   None     None  
2              3   None     None  
3              4   None     None  
4              5   None     None  
